In [1]:
#Data Generation:
import numpy as np
import pandas as pd

np.random.seed(42)
raw_names = [
    '  aArAv sharma ',
    'riya VERMA',
    '  Kavya ',
    'rahul  ',
    np.nan,
    'SNEHA patel',
    'deepak',
    ' Tanvi   ',
]
raw_cities = [
    '  mumbai  ',
    'DELHI',
    'bengaluru',
    np.nan,
    ' pune ',
    'CHENNAI ',
    'hyd',
    'Hyderabad',
]

rows = []
for i in range(1, 101):
    rows.append({
        'customer_id': 1000 + i if i % 15 != 0 else np.nan,
        'name': np.random.choice(raw_names),
        'age': (
            str(np.random.randint(18, 65))
            if i % 10 != 0
            else np.random.choice(['twenty', np.nan])
        ),
        'city': np.random.choice(raw_cities),
        'email': (
            f'user_{i}@example.com' if i % 8 != 0 else f'USER_{i}@EXAMPLE.COM '
        ),
        'purchase_amount': (
            str(np.random.randint(100, 5000)) if i % 12 != 0 else np.nan
        ),
        'rating': np.random.choice([1, 2, 3, 4, 5, np.nan]),
    })

messy_df = pd.DataFrame(rows)
messy_df = pd.concat([messy_df, messy_df.iloc[:8]], ignore_index=True)
messy_df.to_csv('messy_customer_data.csv', index=False)

print('Generated messy dataset. Shape:', messy_df.shape)

Generated messy dataset. Shape: (108, 7)


In [2]:
#Inspection:
df_dirty = pd.read_csv('messy_customer_data.csv')
print('Initial Shape:', df_dirty.shape)
print('\nMissing Values:\n', df_dirty.isnull().sum())
print('\nDuplicate Rows:', df_dirty.duplicated().sum())
df_dirty.head()

Initial Shape: (108, 7)

Missing Values:
 customer_id         6
name               14
age                 5
city               14
email               0
purchase_amount     8
rating             18
dtype: int64

Duplicate Rows: 8


,customer_id,name,age,city,email,purchase_amount,rating
0,1001.0,deepak,46,hyd,user_1@example.com,3872.0,5.0
1,1002.0,deepak,36,hyd,user_2@example.com,4526.0,3.0
2,1003.0,Tanvi,53,Hyderabad,user_3@example.com,230.0,NaN
3,1004.0,NaN,19,Hyderabad,user_4@example.com,2533.0,4.0
4,1005.0,NaN,50,NaN,user_5@example.com,3485.0,NaN


In [3]:
# Cleaning & Export:
# Remove duplicates
df_cleaned = df_dirty.drop_duplicates().copy()

# Clean strings
df_cleaned['name'] = (
    df_cleaned['name'].astype(str).str.strip().str.title().replace('Nan', np.nan)
)
df_cleaned['city'] = (
    df_cleaned['city']
    .astype(str)
    .str.strip()
    .str.title()
    .replace({'Hyd': 'Hyderabad', 'Nan': np.nan})
)
df_cleaned['email'] = df_cleaned['email'].astype(str).str.strip().str.lower()

# Type conversions & Imputations
df_cleaned['age'] = pd.to_numeric(df_cleaned['age'], errors='coerce')
df_cleaned['age'] = (
    df_cleaned['age'].fillna(df_cleaned['age'].median()).astype(int)
)

df_cleaned['purchase_amount'] = pd.to_numeric(
    df_cleaned['purchase_amount'], errors='coerce'
).fillna(0.0)
df_cleaned['rating'] = df_cleaned['rating'].fillna(
    df_cleaned['rating'].mode()[0]
)

# Clean Customer ID
df_cleaned = df_cleaned.dropna(subset=['customer_id'])
df_cleaned['customer_id'] = df_cleaned['customer_id'].astype(int)

# Save cleaned output
df_cleaned.to_csv('cleaned_customer_data.csv', index=False)
print('Cleaning Complete! Final Shape:', df_cleaned.shape)
df_cleaned.head()

Cleaning Complete! Final Shape: (94, 7)


,customer_id,name,age,city,email,purchase_amount,rating
0,1001,Deepak,46,Hyderabad,user_1@example.com,3872.0,5.0
1,1002,Deepak,36,Hyderabad,user_2@example.com,4526.0,3.0
2,1003,Tanvi,53,Hyderabad,user_3@example.com,230.0,5.0
3,1004,NaN,19,Hyderabad,user_4@example.com,2533.0,4.0
4,1005,NaN,50,NaN,user_5@example.com,3485.0,5.0
